# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading, inspecting, and analyzing the FAIR^2 dataset package using the `mlcroissant` library.

### Dataset Source
The dataset source is defined by a [Croissant schema](https://mlcommons.org/croissant/) at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Set the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}\n")
# Show some key metadata fields
print(f"Version: {metadata.version}")
print(f"Date Published: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs (referencing by `@id`).

In [ ]:
# List all available record sets by @id
record_sets = list(dataset.record_sets.values())

if len(record_sets) == 0:
    print("No record sets found in dataset metadata.")
else:
    print("Available Record Sets (@id):\n---------------------------")
    for rs in record_sets:
        print(f"@id: {rs['@id']} | name: {rs.get('name')} | description: {rs.get('description')}")
    print()

# For demonstration, pick the first available record set (if any)
if len(record_sets) > 0:
    chosen_record_set_id = record_sets[0]['@id']
    print(f"Example Record Set @id: {chosen_record_set_id}")
    # List all fields within this record set
    fields = dataset.record_sets[chosen_record_set_id]['field']
    print(f"Fields in '{chosen_record_set_id}':")
    for field in fields:
        if isinstance(field, dict):
            print(f"  - @id: {field['@id']} | name: {field.get('name')} | dataType: {field.get('dataType')}")
        else:
            print(f"  - {field}")
else:
    print("Cannot proceed to inspect fields; no record sets detected.")

## 3. Data Extraction
Load data from available record sets into DataFrames for analysis. All entities are referenced by their `@id`.

In [ ]:
# Automatically extract all (non-empty) record sets by their @id
available_record_set_ids = list(dataset.record_sets.keys())
if not available_record_set_ids:
    print("No record sets available for extraction!")
else:
    dataframes = {}
    for record_set_id in available_record_set_ids:
        print(f"Loading data for record set: {record_set_id}")
        try:
            # records() returns a generator of dicts (records)
            records = list(dataset.records(record_set=record_set_id))
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"  Columns: {dataframes[record_set_id].columns.tolist()}")
            print(f"  Sample:\n{dataframes[record_set_id].head(2)}\n")
        except Exception as e:
            print(f"  Could not load records: {e}")

    # For subsequent analysis, automatically pick first available DataFrame
    if dataframes:
        # Pick first loaded record set as an example
        main_record_set_id = next(iter(dataframes.keys()))
        print(f"Main record set selected for EDA: {main_record_set_id}")
    else:
        main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering records, normalizing numeric fields, and grouping data. All columns referenced by their `@id`.

In [ ]:
# EDA: Select a numeric field for filtering and normalization using @id
import numpy as np

if main_record_set_id is None:
    print("No loaded DataFrame available for EDA.")
else:
    df = dataframes[main_record_set_id]
    print(f"Columns in main DataFrame (@id): {df.columns.tolist()}")

    # Try to auto-detect a numeric field (float/int dtype)
    numeric_candidate = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_candidate = col
            break

    if numeric_candidate is None:
        print("No numeric field found to apply filtering and normalization.")
    else:
        print(f"Using numeric field: {numeric_candidate} (@id)")

        # Example filter: keep values above 10 (arbitrary threshold)
        threshold = 10
        filtered_df = df[df[numeric_candidate] > threshold].copy()

        print(f"Filtered records ({numeric_candidate} > {threshold}): {len(filtered_df)} rows")
        display_cols = [numeric_candidate] + [col for col in df.columns if col != numeric_candidate][:2]  # Some context columns
        print(filtered_df[display_cols].head())

        # Normalize the numeric field (standard score)
        col_norm = f"{numeric_candidate}_normalized"
        filtered_df[col_norm] = (filtered_df[numeric_candidate] - filtered_df[numeric_candidate].mean()) / filtered_df[numeric_candidate].std()
        print(f"\nNormalized '{numeric_candidate}':")
        print(filtered_df[[numeric_candidate, col_norm]].head())

        # Try to group by a string/categorical field
        group_candidate = None
        for col in df.columns:
            if (col != numeric_candidate) and (df[col].dtype == object or str(df[col].dtype).startswith('string')):
                if df[col].nunique() < (len(df) // 2):  # Skip if too many unique values
                    group_candidate = col
                    break
        if group_candidate:
            grouped = filtered_df.groupby(group_candidate)[numeric_candidate].mean().reset_index()
            print(f"\nGrouped mean {numeric_candidate} by {group_candidate} (@id):")
            print(grouped.head())
        else:
            print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset, referencing columns by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_candidate is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_candidate].dropna(), bins=30, kde=True)
    plt.title(f"Histogram of {numeric_candidate} (@id)")
    plt.xlabel(numeric_candidate)
    plt.ylabel("Count")
    plt.show()

    # If grouping field was found in EDA, show boxplot
    if 'group_candidate' in locals() and group_candidate:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=filtered_df, x=group_candidate, y=numeric_candidate)
        plt.title(f"{numeric_candidate} by {group_candidate} (@id)")
        plt.ylabel(numeric_candidate)
        plt.xlabel(group_candidate)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric or grouping field for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR^2 rangeland management dataset using the `mlcroissant` library. Following Croissant conventions, all record sets, fields, and columns were referenced by their `@id`. We inspected dataset metadata, extracted records as DataFrames, completed basic EDA, and visualized distributions, laying the groundwork for more advanced machine learning or policy analysis on knowledge adoption in Northern Kenya. For further insights, see the [dataset documentation](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) or explore additional fields using their `@id` values.